In [1]:
from datetime import datetime, timedelta, timezone
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MetaTrader5 as mt5

In [ ]:
mt5.initialize()

login = os.environ["FTMO_DEMO_LOGIN"]
password = os.environ["FTMO_DEMO_PASSWORD"]
server = os.environ["FTMO_DEMO_SERVER"]

mt5.login(login=login, password=password, server=server)

In [ ]:
gmt_plus_2 = timezone(timedelta(hours=2))

symbol = "BTCUSD"

minutes = 60

start_time = datetime.now(tz=gmt_plus_2).replace(tzinfo=timezone.utc) - timedelta(minutes=minutes)
end_time = datetime.now(tz=gmt_plus_2).replace(tzinfo=timezone.utc)

ticks = mt5.copy_ticks_range(symbol, start_time, end_time, mt5.COPY_TICKS_ALL)
df_ticks = pd.DataFrame(ticks)
df_ticks['datetime'] = pd.to_datetime(df_ticks['time_msc'], unit='ms')
#df_ticks['datetime'] = df_ticks['datetime'].dt.tz_localize('Etc/GMT-2').dt.tz_convert('UTC')
df_ticks.set_index('datetime', inplace=True)


df_ticks.bid.plot(linewidth=0.5)
df_ticks.ask.plot(linewidth=0.5)


In [ ]:

# Initialize an empty DataFrame to store tick data
df_tick = pd.DataFrame()
last_tick_time = None

# Set up the plot
plt.ion()
fig, ax = plt.subplots()
line_bid, = ax.plot([], [], label='Bid Price', color='blue', linewidth=0.5)
line_ask, = ax.plot([], [], label='Ask Price', color='red', linewidth=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Price')
ax.legend()

start_time = datetime.fromisoformat("2024-12-22T17:15Z")
while True:
    # Request tick data
    ticks = mt5.copy_ticks_from(symbol, start_time, -1, mt5.COPY_TICKS_ALL)
    if ticks is None or len(ticks) == 0:
        print(f"Failed to get tick data for symbol {symbol}, error code:", mt5.last_error())
    else:
        # Convert ticks to DataFrame and append to existing DataFrame
        df_new = pd.DataFrame(ticks)
        # Mark the first line of df_new by setting its 'volume' column value to 1
        print(last_tick_time)

        if not df_new.empty:
            df_new['datetime'] = pd.to_datetime(df_new['time_msc'], unit='ms')
            df_new.set_index('datetime', inplace=True)

            # Merge the new data into the existing DataFrame to ensure unique index
            df_tick = df_tick.combine_first(df_new)

            # Update start_time to the last processed tick time
            last_tick_time = df_new.index[-1]
            start_time = last_tick_time.to_pydatetime()

            # Update the plot
            line_bid.set_data(df_tick.index, df_tick['bid'])
            line_ask.set_data(df_tick.index, df_tick['ask'])
            ax.set_ylim([df_tick.bid.min(), df_tick.ask.max()])
            ax.set_xlim([df_tick.index.min(), df_tick.index.max()])
            plt.draw()
            plt.pause(1)

In [ ]:

plt.ion()
fig, ax = plt.subplots()
line_bid, = ax.plot([], [], label='Bid Price', color='blue', linewidth=0.5)
line_ask, = ax.plot([], [], label='Ask Price', color='red', linewidth=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Price')
ax.legend()


line_bid.set_data(df_tick.index, df_tick['bid'])
line_ask.set_data(df_tick.index, df_tick['ask'])
ax.set_ylim([df_tick.bid.min(), df_tick.ask.max()])
ax.set_xlim([df_tick.index.min(), df_tick.index.max()])


In [ ]:

import os
os.chdir("..")
from trader.trader import Trader

trader = Trader()
trader.add_broker('mt5')

trader.add_data("EURUSD", start="2024-12-01 00:00Z", granularities="tick")
#trader.add_indicator('dc', sigma=0.0002, high_colname='bid', low_colname='ask')

fig = trader.plot()


In [10]:
import pandas as pd
from datetime import datetime, timedelta, timezone
import time
df = trader.data[0].df
symbol = "EURUSD"

sessions = [
    ('asia_pacific', '00:00', '11:00', 'Etc/GMT-2', '13826810'),  # Combined Tokyo and Sydney session
    ('london', '10:00', '19:00', 'Etc/GMT-2', '14675921'),
    ('new_york', '15:00', '00:00', 'Etc/GMT-2', '13294079')
]


In [ ]:
sessions_df_list = []
for name, start_time, end_time, timezone, color in sessions:
    session_df = df.between_time(start_time, end_time)
    session_df_summary = session_df.resample('D').agg(
        min=('bid', 'min'),
        max=('bid', 'max'),
        mean=('bid', 'mean'),
        median=('bid', 'median'),
        start_time=('bid', lambda x: x.index.min().round('H')),
        end_time=('bid', lambda x: x.index.max().round('H'))
    )
    session_df_summary.sort_values(by="start_time", inplace=True)
    session_df_summary["name"] = name
    session_df_summary["color"] = color
    sessions_df_list.append(session_df_summary)

sessions_df = pd.concat(sessions_df_list).sort_index().dropna()
sessions_df

In [ ]:
sessions_df.start_time[0].replace(tzinfo=None)

In [ ]:
# Read the file content (replace 'file_path' with your file path)
file_path = r"C:\Users\vynde\AppData\Roaming\MetaQuotes\Terminal\49CDDEAA95A409ED22BD2287BB67CB9C\MQL5\Profiles\Templates\default.tpl"

objects = [
    dict(
    type="20",
    name=f'{symbol} {row["start_time"]} {row["name"]} session',
    color=f"{row['color']}",
    background="1",
    filling="1",
    date1=f"{int(row['start_time'].replace(tzinfo=None).timestamp())}",
    date2=f"{int(row['end_time'].replace(tzinfo=None).timestamp())}",
    value1=f"{row['min']}",
    value2=f"{row['max']}"
    )

    for i, row in sessions_df.iterrows()
]

for obj in objects:
    print("<object>")
    for key, value in obj.items():
        print(f"{key}={value}")
    print("</object>")
    print("")